# GPU evaluation: vLLM + Qwen3-8B, RAGAS, MLflow sweep

Runs on **Kaggle, GPU T4 x2, Internet ON**. What it produces:

1. **Qwen3-8B served by vLLM** (OpenAI-compatible API, tensor-parallel over both T4s)
2. **MLflow sweep**: 5 chunking configs x the 20-question CI subset, every run logged
3. **Full RAGAS run**: best config x all 54 questions x 4 metrics, registered in the MLflow Model Registry as `civil-code-rag-chunking@production`
4. `gpu_eval_results.zip` to download: MLflow database + artifacts, per-question reports, `reports/ragas_results.json`

**Before running:** upload `data/interim/civil_code.json` from your laptop as a private Kaggle Dataset and add it to this notebook (*Add Input*). Budget: about 2-3 GPU-hours.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code and corpus

In [ ]:
!git clone -q https://github.com/ZeinabMahfouz/The-Egyptian-Civil-Code-RAG.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -1
# Kaggle mounts datasets at varying depths, so find the file rather than guess the path.
!mkdir -p data/interim eval_out && cp "$(find /kaggle/input -name civil_code.json | head -1)" data/interim/ && ls -la data/interim

In [ ]:
# Must match the corpus your DVC pipeline built, or the CI gate will call the evaluation stale.
!md5sum data/interim/civil_code.json
!grep -A3 "path: data/interim/civil_code.json" dvc.lock | grep "md5:"

## 2. vLLM in its own environment

vLLM pins its own torch/transformers, and RAGAS pins `openai<2` through `instructor`. Same lesson as guardrails-ai: separate environments, so the two never fight. Kaggle's `python -m venv` has no ensurepip, so the venv is made with `uv`.

In [ ]:
!pip install -q uv
!uv venv /kaggle/vllm-env --python 3.11
!uv pip install -q --python /kaggle/vllm-env/bin/python vllm wrapt
!/kaggle/vllm-env/bin/python -c "import vllm, torch; print('vllm', vllm.__version__, 'torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# Qwen3-8B in fp16 (T4 has no bf16), split across both GPUs.
# 0.75 GPU memory leaves room for BGE-M3 (indexing on cuda:1, RAGAS embeddings on cuda:0).
# Kaggle doesn't allow background shell jobs (`!cmd &`), so start it with Popen.
import os, subprocess
vllm_proc = subprocess.Popen(
    ["/kaggle/vllm-env/bin/vllm", "serve", "Qwen/Qwen3-8B",
     "--tensor-parallel-size", "2", "--dtype", "half", "--max-model-len", "8192",
     "--gpu-memory-utilization", "0.75", "--port", "8000"],
    stdout=open("/kaggle/working/vllm.log", "w"), stderr=subprocess.STDOUT,
    env={**os.environ, "PYTHONPATH": ""}, start_new_session=True,
)

In [ ]:
# First start downloads ~16 GB of weights; wait until the API answers.
import time, urllib.request
for i in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
        print(f"vLLM ready after ~{i*15}s"); break
    except Exception:
        time.sleep(15)
else:
    print("not ready -- check the log below")
!tail -n 20 /kaggle/working/vllm.log

## 3. Evaluation environment

In [ ]:
!pip install -q "ragas==0.4.3" "langchain-openai==1.1.9" "langchain-huggingface==1.2.2" \
    "qdrant-client==1.19.1" "mlflow==3.16.1" "openai==1.109.1" "instructor==1.3.2" \
    sentence-transformers pyyaml langfuse==4.15.6 prometheus-client==0.26.0
!pip install -q -e . --no-deps

In [ ]:
# Smoke test: one real question through retrieval-free generation on vLLM.
import sys; sys.path.insert(0, "src")
from egyptian_civil_code_rag.backends import openai_backend
gen = openai_backend("http://localhost:8000/v1", "Qwen/Qwen3-8B")
t = time.time()
print(gen("In one sentence: what does Article 147 of the Egyptian Civil Code establish?"))
print(gen.last_usage, f"{time.time()-t:.1f}s")

## 4. MLflow sweep: 5 chunking configs x 20 questions

Each config gets its own index (same chunking and embedding code as `dvc repro`), its own answers from Qwen3-8B, and its own RAGAS scores. Expect about 15-25 min per config.

In [ ]:
# Output goes to a log file: the Kaggle display can freeze on long outputs (reload the page if so).
!python scripts/gpu_eval.py sweep --embed-device cuda:1 > eval_out/sweep.log 2>&1; grep "\[sweep\]" eval_out/sweep.log

In [ ]:
import json, mlflow, pandas as pd
mlflow.set_tracking_uri("sqlite:////kaggle/working/repo/eval_out/mlflow.db")
runs = mlflow.search_runs(experiment_names=["civil-code-rag-chunking"])
cols = ["params.config", "params.chunk_size", "params.n_chunks", "metrics.faithfulness",
        "metrics.context_precision", "metrics.context_recall", "metrics.answer_relevancy",
        "metrics.article_hit_rate", "metrics.faithfulness_n_scored"]
display(runs[[c for c in cols if c in runs.columns]].sort_values("metrics.faithfulness", ascending=False))
print(json.load(open("eval_out/sweep_summary.json"))["best"])

## 5. Full run on the best config: all 54 questions, all 4 metrics, registry

In [ ]:
!python scripts/gpu_eval.py full --embed-device cuda:1 > eval_out/full.log 2>&1; tail -n 3 eval_out/full.log

In [ ]:
report = json.load(open("eval_out/reports/ragas_results.json"))
print(json.dumps(report["meta"], indent=2, ensure_ascii=False)[:1500])
print(json.dumps(report["aggregate"], indent=2))
df = pd.DataFrame(report["rows"])
display(df.groupby(["_category", "_language"])[["faithfulness", "context_precision", "context_recall", "answer_relevancy"]].mean().round(3))

In [ ]:
# CI gate preview: what scripts/ragas_gate.py will say once this report is committed.
!cp eval_out/reports/ragas_results.json reports/ragas_results.json
!python scripts/ragas_gate.py

## 6. Download the results

In [ ]:
!cd /kaggle/working/repo && zip -qr /kaggle/working/gpu_eval_results.zip eval_out/mlflow.db mlruns eval_out/reports eval_out/sweep_summary.json eval_out/*.log
%cd /kaggle/working
from IPython.display import FileLink
display(FileLink("gpu_eval_results.zip"))  # click to download, then follow the README steps